# 05 -- Stage C: joint fine-tune (gate + full model)

Instantiates the gate, unfreezes the encoder per `training.stage_c_unfreeze`, and trains with `CE(combined_probs, y) + lambda_balance * load_balance_penalty + lambda_dataset_aux * CE(gate_weights, dataset_id)`. `training.stage_c.gate_supervision` (`none`/`light_aux`/`hard`/`damex`) selects the router objective. In `damex`, task weights are detached so dataset CE and balancing are the router's only signals; the `moe_dataset_damex` preset also uses dataset-owned expert updates.


## Setup

Run this cell first. It's the ONLY cell you should need to edit: change
`CONFIG_OVERRIDES` (a list of `--set key.path=value` style dotted overrides,
same syntax as `training.run`'s CLI) to narrow `data.active_datasets`,
switch `architecture`, point at a different Drive folder, etc.


In [ ]:
# ---- Single config cell: this is the only cell you should need to edit ----
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess, os
    REPO_DIR = '/content/dataset_moe_nids'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/selimsidan/dataset_moe_nids', REPO_DIR], check=True)
    %cd $REPO_DIR
    %pip install -q -r requirements.txt

CONFIG_PATH = 'config/default.yaml'

# Dotted --set overrides, same syntax as training.run's CLI. Narrow
# data.active_datasets to 2-3 datasets here for a fast iteration cycle --
# every downstream module (registry, harmonizer, expert-bank sizing,
# checkpoints) adapts automatically, no other code changes needed.
CONFIG_OVERRIDES = [
    # 'data.active_datasets=[NF-UNSW-NB15-v3,NF-BoT-IoT-v3]',
    # 'architecture=moe_dataset_soft',
    # 'training.device=cuda',
]

from training.config import load_config
config = load_config(CONFIG_PATH, CONFIG_OVERRIDES)
print('run_name:', config['run_name'])
print('architecture:', config['architecture'])
print('active_datasets:', config['data']['active_datasets'])
print('checkpoint_dir:', config['training']['checkpoint_dir'])


In [ ]:
from training.dataset import prepare_datasets
from training.stage_c_jointfinetune import run_stage_c
from evaluation.metrics import evaluate_per_dataset, evaluate_predictions
import torch

data = prepare_datasets(config)
model = run_stage_c(config, data)

model.eval()
with torch.no_grad():
    scores = model(torch.from_numpy(data.test.features))['combined_probs'].numpy()
    preds = scores.argmax(axis=1)
result = evaluate_predictions(data.test.class_idx, preds, data.class_names, scores)
per_dataset = evaluate_per_dataset(data.test.class_idx, preds, data.test.dataset_name, data.class_names, scores)

print(f"macro_p/r/f1={result.macro_precision:.4f}/{result.macro_recall:.4f}/{result.macro_f1:.4f} micro_p/r/f1={result.micro_precision:.4f}/{result.micro_recall:.4f}/{result.micro_f1:.4f} weighted_p/r/f1={result.weighted_precision:.4f}/{result.weighted_recall:.4f}/{result.weighted_f1:.4f} roc_auc_ovr_macro={result.roc_auc_ovr_macro:.4f}")
for name, r in per_dataset.items():
    print(f'  {name:20s} macro_f1={r.macro_f1:.4f} weighted_f1={r.weighted_f1:.4f} roc_auc_ovr_macro={r.roc_auc_ovr_macro:.4f}')
